# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring the FAIR² mass spectrometry dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL: [https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json)

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata_obj = dataset.metadata

print(f"Name: {metadata_obj.name}")
print(f"Description: {metadata_obj.description}")
print(f"Published: {metadata_obj.datePublished}")
print(f"Version: {metadata_obj.version}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

We'll enumerate and inspect the record sets available in the dataset, and for each record set, list its fields and columns using their `@id`s as required by the Croissant specification.

In [ ]:
# Show all available record sets and their fields, referencing all by @id.
record_sets = list(dataset.record_sets())
print(f"Number of record sets: {len(record_sets)}")
record_set_ids = []
for rec in record_sets:
    print(f"\nRecord Set @id: {rec['@id']}")
    record_set_ids.append(rec['@id'])
    print(f"  Name: {rec.get('name', None)}")
    if 'field' in rec:
        field_ids = rec['field'] if isinstance(rec['field'], list) else [rec['field']]
        print(f"  Fields/Columns: {[field['@id'] if isinstance(field, dict) and '@id' in field else field for field in field_ids]}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. We will use the record set and field `@id`s discovered above.

For this notebook, we will (by inspection of the schema) extract all record sets found in the dataset. If the dataset has only one main record set (which is often the case), it will be evidenced above.

In [ ]:
# Collect all discovered record set @ids
print(f"Using record set IDs: {record_set_ids}")
dataframes = {}
for rsid in record_set_ids:
    try:
        print(f"\nReading rows from record set: {rsid}")
        records = list(dataset.records(record_set=rsid))
        df = pd.DataFrame(records)
        print(f"Loaded {len(df)} records for {rsid}.")
        dataframes[rsid] = df
        print(f"Columns for {rsid}: {df.columns.tolist()}")
        print(df.head())
    except Exception as e:
        print(f"Error loading {rsid}: {e}")

## 4. Exploratory Data Analysis (EDA)
Apply data processing steps: filtering, normalization, categorization, and grouping. All field references use their `@id`s.

*Please note:* To proceed, we need to select a numeric field and a group field (typically, these are provided by inspecting the field IDs and data in the DataFrame). For this example, we will attempt to use the first discovered record set and search for numeric and grouping fields.

In [ ]:
# Attempt EDA on the first record set
import numpy as np

def find_numeric_and_group_fields(df):
    # Prefer columns with float or int types
    numeric_cols = df.select_dtypes(include=[np.number]).columns
    group_cols = [col for col in df.columns if 'type' in col.lower() or 'cat' in col.lower() or 'group' in col.lower() or df[col].nunique() < min(10, len(df)//5)]
    if len(numeric_cols) > 0:
        numeric_field = numeric_cols[0]
    else:
        numeric_field = df.columns[0] # fallback
    group_field = group_cols[0] if group_cols else df.columns[0] # fallback
    return numeric_field, group_field

if len(dataframes) > 0:
    record_set_id = list(dataframes.keys())[0]
    df = dataframes[record_set_id]
    print(f"Selected record set for EDA: {record_set_id}")
    if df.shape[0] == 0:
        print("No data loaded for this record set.")
    else:
        numeric_field, group_field = find_numeric_and_group_fields(df)
        print(f"Chosen numeric field (@id): {numeric_field}")
        print(f"Chosen group field (@id): {group_field}")

        # Filtering
        if pd.api.types.is_numeric_dtype(df[numeric_field]):
            threshold = df[numeric_field].mean()
            filtered_df = df[df[numeric_field] > threshold]
            print(f"Filtered records with {numeric_field} > mean ({threshold:.2f}):")
            print(filtered_df.head())

            # Normalization
            filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
            print(f"Normalized {numeric_field} for filtered records:")
            print(filtered_df[[numeric_field, f"{numeric_field}_normalized"].copy()].head())

            # Grouping
            if group_field in df.columns and pd.api.types.is_object_dtype(df[group_field]):
                grouped_df = filtered_df.groupby(group_field)[numeric_field].mean()
                print(f"Grouped mean {numeric_field} by {group_field}:")
                print(grouped_df.head())
        else:
            print(f"Field {numeric_field} is not numeric; skipping filter/normalize/group.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset. Replace field names as appropriate for your data.

If the selected numeric field has more than 0 data points, plot its distribution and its mean grouped by the chosen group field.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if len(dataframes) > 0:
    df = dataframes[record_set_id]
    if df.shape[0] > 0 and pd.api.types.is_numeric_dtype(df[numeric_field]):
        fig, ax = plt.subplots(1, 2, figsize=(12,4))
        sns.histplot(df[numeric_field], bins=30, kde=True, ax=ax[0])
        ax[0].set_title(f"Distribution of {numeric_field}")
        ax[0].set_xlabel(numeric_field)

        if group_field in df.columns and pd.api.types.is_object_dtype(df[group_field]):
            group_means = df.groupby(group_field)[numeric_field].mean().sort_values()
            group_means.plot(kind='bar', ax=ax[1])
            ax[1].set_title(f"Mean {numeric_field} by {group_field}")
            ax[1].set_ylabel(f"Mean of {numeric_field}")

        plt.tight_layout()
        plt.show()
    else:
        print(f"Cannot visualize field {numeric_field}; insufficient numeric data.")

## 6. Conclusion
In this notebook, we loaded the FAIR² mass spectrometry dataset Croissant schema, listed all record sets and fields by their `@id`, extracted and loaded records, and demonstrated exploratory analysis and visualization using numeric and group fields referenced by their Croissant `@id`s.

*Key findings*:
- The dataset structure, fields, and primary data types can be systematically explored and loaded with `mlcroissant`.
- Filtering, normalization and grouping can be efficiently performed using pandas once loaded.
- Plots reveal the distribution of quantitative experimental parameters, enabling biological and statistical interpretation.

For specific field definitions or domain interpretations, consult the dataset metadata or associated publication.